# Review — 4_langchain_langgraph (100% local, with Ollama)

Review notebook of the **design patterns** covered in `1_lab1` → `5_lab5` (LangChain, LangGraph,
`create_agent`, Deep Agents, Sidekick), rewritten to run entirely on **free Ollama models**
(no OpenAI / Serper / Pushover keys, no LangSmith).

## Setup

```bash
ollama pull qwen3:8b   # reliable tool-calling / structured-output model
ollama serve           # if not already running
pip install ddgs       # free web search, replaces GoogleSerperRun
```

Everything else (`langchain`, `langgraph`, `deepagents`, `langgraph-checkpoint-sqlite`, `wikipedia`)
is already in the course environment.

> **Only one thing changes vs. the labs:** the model. `ChatOpenAI(base_url=".../v1")` points at
> Ollama (the lab1 "any OpenAI-compatible provider" trick), and that object is passed wherever the
> labs pass `"openai:gpt-5.4-mini"`. `reasoning_effort="none"` turns off qwen3's thinking:
> ~15x faster per call, and plenty for these exercises.

## The four layers (lab1)

| Layer | Package | You control |
|---|---|---|
| 1. Building blocks | `langchain-core` + `langchain-openai` | everything, tool loop by hand |
| 2. Orchestration | `langgraph` | the graph (nodes, edges, state) |
| 3. Agent | `langchain` (`create_agent`) | model, tools, prompt, middleware |
| 4. Harness | `deepagents` | your intent |

## Index of macro-blocks

| # | Block | Pattern | Source lab |
|---|-------|---------|------------|
| 1 | Chat model | `invoke` / `stream`, messages, any provider | lab1 |
| 2 | Tools by hand | `@tool`, `bind_tools`, `.tool_calls`, `ToolMessage` loop | lab1 |
| 3 | Structured output | `with_structured_output(BaseModel)` | lab1 |
| 4 | Graph mechanics | State + reducer, 5 steps, no LLM needed | lab2 |
| 5 | Tool loop as a graph | `ToolNode` + `tools_condition`, extra LLM node | lab2 |
| 6 | Checkpointing | `MemorySaver` / `SqliteSaver`, `thread_id`, time travel | lab2 |
| 7 | `create_agent` | The Block 5 graph in one line; memory; `response_format` | lab3 |
| 8 | Middleware | `@wrap_tool_call` + the prebuilt catalogue | lab3 / lab5 |
| 9 | Human-in-the-loop | `interrupt` → `Command(resume=...)`: approve / edit / reject | lab5 |
| 10 | Deep Agents | Todo planning, filesystem, sub-agents | lab4 |
| 11 | Sidekick | Worker + evaluator loop against success criteria | lab5 |

Skipped (need Node/browser or paid keys): Playwright MCP browser (see `6_mcp`), Agent Skills + slide,
Gradio UIs, LangSmith tracing.

In [ ]:
import os, json, random, requests
from typing import Annotated
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown, display

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

OLLAMA = "http://localhost:11434"
llm = ChatOpenAI(model="qwen3:8b", base_url=f"{OLLAMA}/v1", api_key="ollama", reasoning_effort="none")

def check_model(name):
    try:
        models = {m["id"] for m in requests.get(f"{OLLAMA}/v1/models", timeout=3).json().get("data", [])}
    except Exception:
        print("Ollama unreachable on localhost:11434 — make sure 'ollama serve' is running.")
        return False
    if not any(m == name or m.startswith(name + ":") for m in models):
        print(f"Model '{name}' not found. Run: ollama pull {name}")
        return False
    return True

check_model("qwen3:8b")

## Block 1 — Chat model: invoke, stream, messages

**Pattern:** a chat model object with one key method, `invoke`. `stream` yields chunks for a
live feel. Input can be a string, LangChain message objects (`SystemMessage`, `HumanMessage`,
`AIMessage`) or the usual list of dicts — all equivalent.

Any OpenAI-compatible endpoint works with `ChatOpenAI(base_url=...)` (that's how we use Ollama here);
native packages exist too (`langchain-anthropic`, `langchain-ollama`, ...).

In [ ]:
print(llm.invoke("In 1 sentence, what does it mean for an AI Agent to be autonomous").content)

for chunk in llm.stream("Tell me a two line poem about autonomous agents"):
    print(chunk.content, end="", flush=True)
print()

messages = [SystemMessage("You are a terse assistant who answers in exactly five words."),
            HumanMessage("What is the capital of France?")]
print(llm.invoke(messages).content)
print(llm.invoke([{"role": "system", "content": "You are a terse assistant who answers in exactly five words."},
                  {"role": "user", "content": "What is the capital of France?"}]).content)

## Block 2 — Tools by hand (layer 1)

**Pattern:** `@tool` builds the schema from type hints + docstring (like `@function_tool` in
`2_openai`). `llm.bind_tools([...])` tells the model the tools exist; the reply may then carry
`.tool_calls` instead of text. At layer 1 **you** run the loop — the same loop as `1_foundations`,
with `ToolMessage` instead of `{"role": "tool"}` dicts.

In [ ]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

print(get_share_price.name, "|", get_share_price.description, "|", get_share_price.args)

llm_with_tools = llm.bind_tools([get_share_price])

conversation = [HumanMessage("What is the share price of Amazon?")]
ai_message = llm_with_tools.invoke(conversation)
print("tool_calls:", ai_message.tool_calls)
conversation.append(ai_message)

for call in ai_message.tool_calls:
    result = get_share_price.invoke(call["args"])
    conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

print(llm_with_tools.invoke(conversation).content)

## Block 3 — Structured output

**Pattern:** `llm.with_structured_output(Model)` returns a runnable whose `invoke` gives back a
**Pydantic object**. Same idea as `output_type=` (OpenAI Agents SDK) and `output_pydantic=` (CrewAI).

In [ ]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

company = llm.with_structured_output(Company).invoke("Tell me about Amazon the technology company")
print(company, "| just the ticker:", company.ticker)

## Block 4 — Graph mechanics, no LLM needed (layer 2)

**Pattern — five steps, always the same:** define the **State** → `StateGraph(State)` →
`add_node` → `add_edge` → `compile()`.

- **State** is a `TypedDict` passed from node to node. Each **node** is a plain function
  `state -> dict of updates`.
- By default an update **replaces** the field. A **reducer** (attached with `Annotated`) says how to
  combine old + new instead: `add_messages` **appends**, which is what a chat needs.
- LangGraph is agnostic to what runs inside a node — here, no LLM at all.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

def silly_node(state: State) -> dict:
    sentence = f"{random.choice(['Cabbages', 'Unicorns', 'Toasters'])} are {random.choice(['smelly', 'sparkly', 'haunted'])}"
    return {"messages": [{"role": "assistant", "content": sentence}]}

builder = StateGraph(State)
builder.add_node("silly", silly_node)
builder.add_edge(START, "silly")
builder.add_edge("silly", END)
graph = builder.compile()

print(graph.invoke({"messages": [{"role": "user", "content": "say something"}]})["messages"][-1].content)

In [ ]:
# Same five steps, now the node calls the model
def chatbot_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))   # rendered via mermaid.ink (free, needs internet)
print(graph.invoke({"messages": [{"role": "user", "content": "What is a directed graph, in one sentence?"}]})["messages"][-1].content)

## Block 5 — The tool loop as a graph

**Pattern:** two prebuilt pieces turn Block 2's hand-written loop into a graph:

- `ToolNode(tools)` — a node that runs whatever tools the last AI message asked for
- `tools_condition` — a router: go to `"tools"` if there are tool calls, else `END`

Wiring: `START → chatbot`, conditional edge `chatbot → tools | END`, edge `tools → chatbot` (the loop).

Tools can be off-the-shelf (`GoogleSerperRun` from `langchain-community` in the lab) or
homemade `@tool` — they plug in the same way. Here: free DuckDuckGo search + a `print` for Pushover.

**Extra LLM node:** passing a mapping to `add_conditional_edges` reroutes the "finished" branch
to a `translator` node, which writes a **second state field** (`spanish`, no reducer → overwritten).

In [ ]:
from ddgs import DDGS
from langgraph.prebuilt import ToolNode, tools_condition

@tool
def web_search(query: str) -> str:
    """Search the web and return the top results (title, snippet) as text."""
    return "\n".join(f"- {h['title']}: {h['body']}" for h in DDGS().text(query, max_results=5))

@tool
def send_push_notification(text: str) -> str:
    """Send a short push notification to the user's phone."""
    print(f"[PUSH] {text}")   # replaces the Pushover HTTP call
    return "Notification sent"

tools = [web_search, send_push_notification]
llm_with_tools = llm.bind_tools(tools)

class State(TypedDict):
    messages: Annotated[list, add_messages]
    spanish: str

def chatbot_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

def translator_node(state: State) -> dict:
    last = state["messages"][-1].content
    return {"spanish": llm.invoke(f"Translate this into Spanish, replying with the translation only:\n\n{last}").content}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("tools", "chatbot")
builder.add_edge("translator", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))
result = graph.invoke({"messages": [{"role": "user", "content":
    "Use your search tool to find the ingredients in banoffee pie, then send me a push notification with them."}]})
print(result["messages"][-1].content)
print(result["spanish"])

## Block 6 — Checkpointing: memory, persistence, time travel

**Pattern:** one `invoke` = one run of the graph (a sequence of **super-steps**). The reducer
accumulates *within* a run; the next `invoke` starts fresh — the illusion of memory again.

Fix: `compile(checkpointer=...)` saves a **snapshot after every super-step**, filed under a
`thread_id` passed in `config`. Same `thread_id` next time → history reloaded.

- `MemorySaver()` — in RAM; `SqliteSaver.from_conn_string("memory.db")` — on disk, survives restarts
- `graph.get_state(config)` / `graph.get_state_history(config)` — inspect every snapshot
- put a `checkpoint_id` in the config → **resume from that exact moment** (time travel / branching)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

graph = builder.compile(checkpointer=MemorySaver())
config = {"configurable": {"thread_id": "conversation-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Giulio."}]}, config)
print(graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)["messages"][-1].content)

with SqliteSaver.from_conn_string("memory.db") as sql_memory:
    durable = builder.compile(checkpointer=sql_memory)
    sql_config = {"configurable": {"thread_id": "conversation-2"}}
    durable.invoke({"messages": [{"role": "user", "content": "Remember that my favorite color is orange."}]}, sql_config)
    print(durable.invoke({"messages": [{"role": "user", "content": "What is my favorite color?"}]}, sql_config)["messages"][-1].content)

In [ ]:
history = list(graph.get_state_history(config))   # newest first
for h in reversed(history):
    queued = ", ".join(t.name for t in h.tasks) or "(run complete)"
    print(f"step {h.metadata['step']:>2}  {h.metadata['source']:<5}  messages={len(h.values.get('messages', []))}  next: {queued}")

# Time travel: branch from a checkpoint in the middle of the history
earlier = history[len(history) // 2]
replay_config = {"configurable": {"thread_id": "conversation-1",
                                  "checkpoint_id": earlier.config["configurable"]["checkpoint_id"]}}
again = graph.invoke({"messages": [{"role": "user", "content": "What do you know about me?"}]}, replay_config)
print(again["messages"][-1].content)

## Block 7 — `create_agent`: the Block 5 graph in one line (layer 3)

**Pattern:** `create_agent(model, tools, system_prompt)` builds the agent loop for you — and it
returns a **compiled LangGraph graph**: draw it and you see Block 5's `model ⇄ tools` loop.
So everything from layer 2 still applies:

- `checkpointer=` + `thread_id` → memory
- `response_format=Model` → typed result in `result["structured_response"]`
- `invoke` / `ainvoke` (async twin, needed with async tools such as MCP)

`model` can be a string (`"openai:gpt-5.4-mini"`) or a chat model object — here our Ollama `llm`.

In [ ]:
from langchain.agents import create_agent

@tool
def get_weather(city: str) -> str:
    """Return today's weather for a city."""
    return {"London": "rainy, 14 degrees", "Rome": "sunny, 27 degrees"}.get(city, "clear, 20 degrees")

@tool
def get_population(city: str) -> str:
    """Return the population of a city."""
    return {"London": "8.9 million", "Rome": "2.8 million"}.get(city, "unknown")

agent = create_agent(model=llm, tools=[get_weather, get_population],
                     system_prompt="You are a travel assistant. Use your tools to answer questions about cities.")
display(Image(agent.get_graph().draw_mermaid_png()))
print(agent.invoke({"messages": [{"role": "user", "content": "What is the weather and population of Rome?"}]})["messages"][-1].content)

In [ ]:
memory_agent = create_agent(model=llm, tools=[get_weather], checkpointer=MemorySaver(),
                            system_prompt="You are a travel assistant. Use your tools to answer questions about cities.")
config = {"configurable": {"thread_id": "trip-planning"}}
memory_agent.invoke({"messages": [{"role": "user", "content": "I am planning a trip to London."}]}, config)
print(memory_agent.invoke({"messages": [{"role": "user", "content": "What is the weather like where I am going?"}]}, config)["messages"][-1].content)

class CityReport(BaseModel):
    city: str = Field(description="The city name")
    weather: str = Field(description="A short weather description")
    population: str = Field(description="The population")

report_agent = create_agent(model=llm, tools=[get_weather, get_population], response_format=CityReport,
                            # an 8B model fills the schema straight away and invents the values unless told the order
                            system_prompt="First call get_weather and get_population for the city. Only after you have "
                                          "their results, fill in the report with exactly those values.")
report = report_agent.invoke({"messages": [{"role": "user", "content": "Give me a report on London."}]})["structured_response"]
print(report)

## Block 8 — Middleware

**Pattern:** hooks that run *your* code at fixed points of the agent loop: before/after the model
call, or **around each tool call**. The agent-level equivalent of guardrails, but for every step.

Custom: `@wrap_tool_call` (or subclass `AgentMiddleware`, like the Sidekick's `TolerateToolErrors`).
Prebuilt catalogue used in the Sidekick:

| Middleware | Does |
|---|---|
| `TodoListMiddleware` | adds a `write_todos` planning tool; plan exposed in state `todos` |
| `PIIMiddleware("email")` | redacts PII in input (and tool results with `apply_to_tool_results=True`) |
| `ModelCallLimitMiddleware(run_limit=30)` | caps model calls → no runaway loops / bills |
| `SummarizationMiddleware` | compresses long histories to fit the context window |
| `HumanInTheLoopMiddleware` | pauses before chosen tools (Block 9) |

In [ ]:
from langchain.agents.middleware import wrap_tool_call, ModelCallLimitMiddleware

@wrap_tool_call
def log_tool_calls(request, handler):
    call = request.tool_call
    print(f"  [middleware] calling {call['name']} with {call['args']}")
    return handler(request)   # let the call proceed

watched_agent = create_agent(model=llm, tools=[get_weather, get_population],
                             system_prompt="You are a travel assistant. Use your tools.",
                             middleware=[log_tool_calls, ModelCallLimitMiddleware(run_limit=10)])
result = watched_agent.invoke({"messages": [{"role": "user", "content": "Weather and population of London and Rome?"}]})
print("\nFinal answer:", result["messages"][-1].content)

## Block 9 — Human-in-the-loop

**Pattern:** `HumanInTheLoopMiddleware(interrupt_on={"tool_name": True})` + a checkpointer.
When the agent wants that tool, the run **pauses** and returns `result["__interrupt__"]` describing
the pending action. Resume on the same `thread_id` with `Command(resume={"decisions": [...]})`:

- `{"type": "approve"}` — run as proposed
- `{"type": "edit", "edited_action": {"name": ..., "args": {...}}}` — run with changed args
- `{"type": "reject", "message": "..."}` — skip; the message goes back to the model

Restrict choices with `{"tool": {"allowed_decisions": ["approve", "reject"]}}`. It works because the
checkpointer froze the graph mid-run (Block 6).

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

@tool
def book_meeting(person: str, day: str) -> str:
    """Book a meeting with a person on a given day."""
    return f"Meeting booked with {person} on {day}."

approval_agent = create_agent(model=llm, tools=[book_meeting], checkpointer=MemorySaver(),
                              system_prompt="You are a scheduling assistant. Use the book_meeting tool.",
                              middleware=[HumanInTheLoopMiddleware(interrupt_on={"book_meeting": True})])

for thread, decision in [("approve-demo", {"type": "approve"}),
                         ("reject-demo", {"type": "reject", "message": "Rejected - no meetings on Fridays"})]:
    config = {"configurable": {"thread_id": thread}}
    result = approval_agent.invoke({"messages": [{"role": "user", "content": "Book a meeting with Sam on Friday."}]}, config)
    print("PAUSED:", result["__interrupt__"][0].value["action_requests"][0]["description"])
    resumed = approval_agent.invoke(Command(resume={"decisions": [decision]}), config)
    print(f"{decision['type'].upper()} →", resumed["messages"][-1].content, "\n")

## Block 10 — Deep Agents (layer 4)

**Pattern:** `create_deep_agent` = `create_agent` + an opinionated harness with three built-ins:

1. **Planning** — a `write_todos` tool; the agent writes and updates its own plan
2. **Filesystem** — `ls` / `read_file` / `write_file` / `edit_file`; notes and deliverables live in files,
   not in the context. `FilesystemBackend(root_dir, virtual_mode=True)` maps the agent's `/` to a real folder.
3. **Sub-agents** — a `task` tool to delegate a focused job to a helper with a **clean context**
   (dicts with `name`, `description`, `system_prompt`, optionally own `tools` / `model` / `skills`)

Use it for long, multi-step, artifact-producing work; for a quick tool call, `create_agent` is enough.
Skills (`SKILL.md` with progressive disclosure) plug into the same harness — see `4_lab4`.

> The harness adds a long system prompt and many tools: an 8B model gets through it (~5 min), but
> it needs a blunt instruction to actually call `write_file` — without "you MUST save ... the task is
> only done once the file is written" it happily delegated and then answered in chat.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

sandbox = os.path.abspath("sandbox")
os.makedirs(sandbox, exist_ok=True)

research_subagent = {
    "name": "vehicle-researcher",
    "description": "Researches a single electric vehicle and returns a short list of facts about it.",
    "system_prompt": "You research one electric vehicle using the web_search tool and return three concise facts "
                     "a fleet buyer would care about: price, range and charging.",
}

lead = create_deep_agent(
    model=llm,
    tools=[web_search],
    system_prompt="You write comparison briefings for a company choosing electric vehicles for its sales fleet. "
                  "Plan with your todo tool. For each vehicle, delegate the research to your vehicle-researcher "
                  "sub-agent. Then you MUST save a markdown comparison, ending with a clear recommendation, "
                  "using your write_file tool. The task is only done once the file is written.",
    subagents=[research_subagent],
    backend=FilesystemBackend(root_dir=sandbox, virtual_mode=True),
)

result = lead.invoke({"messages": [{"role": "user", "content":
    "Compare the Tesla Model Y and the Ford Mustang Mach-E for our 100-car sales fleet. "
    "Write a short markdown comparison with a recommendation to /fleet.md."}]})

# Only the lead's own calls are in its messages: sub-agent work (searches, even the file write) shows up as `task`
print("Tools called:", [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])])
print("Sandbox:", os.listdir(sandbox))
if os.path.exists(os.path.join(sandbox, "fleet.md")):
    display(Markdown(open(os.path.join(sandbox, "fleet.md")).read()))

## Block 11 — The Sidekick: worker + evaluator loop (lab5, trimmed)

**Pattern:** the worker is a plain **layer 3** `create_agent` (tools + middleware + checkpointer);
around it, **our own loop** (no framework needed):

1. worker attempts the task (the success criteria are appended to the request)
2. **evaluator** = a model with structured output (`success_criteria_met`, `user_input_needed`, `feedback`),
   given the reply **and the list of tools called as evidence**
3. met, or a question for the user, or `MAX_ATTEMPTS` → stop; otherwise send the feedback back
   to the worker on the **same thread** and retry

This is the evaluator-optimizer pattern from `1_foundations` (LLM-as-judge), now closing the loop.
The full `sidekick.py` adds a Playwright browser + filesystem via MCP, PII redaction, and HITL on push
notifications — same skeleton.

> **Limit worth remembering:** the evaluator checks the reply *against the criteria*, not against
> reality. In my run the worker named the wrong laureates (2022's instead of 2023's) and the evaluator
> still said "met": criteria like "the reply names the winners" can't catch a wrong fact. Ask for
> evidence in the criteria (e.g. "cites the source it used"), or give the evaluator a tool to check.

In [ ]:
import uuid, wikipedia
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents.middleware import TodoListMiddleware

wikipedia.set_user_agent("agentic-track-course-review")   # Wikimedia rejects the default user agent
wikipedia_lookup = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

MAX_ATTEMPTS = 3

class EvaluatorOutput(BaseModel):
    feedback: str = Field(description="Feedback on the assistant's response")
    success_criteria_met: bool = Field(description="Whether the success criteria have been met")
    user_input_needed: bool = Field(description="True if the assistant has a question, needs clarification, or is stuck")

worker = create_agent(
    model=llm,
    tools=[web_search, wikipedia_lookup, send_push_notification],
    system_prompt="You are Sidekick, a helpful personal assistant. Use your tools to complete the task. "
                  "Keep working until the success criteria are met, or ask the user a plain question if you need to.",
    middleware=[TodoListMiddleware(), ModelCallLimitMiddleware(run_limit=15)],
    checkpointer=MemorySaver(),
)
evaluator = llm.with_structured_output(EvaluatorOutput)

def run_turn(task: str, success_criteria: str) -> str:
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    message = f"{task}\n\nThe success criteria for this task are: {success_criteria}"
    for attempt in range(1, MAX_ATTEMPTS + 1):
        result = worker.invoke({"messages": [{"role": "user", "content": message}]}, config)
        reply = result["messages"][-1].content
        tools_used = [c["name"] for m in result["messages"] for c in (getattr(m, "tool_calls", None) or [])]
        verdict = evaluator.invoke(
            f"Task: {task}\nSuccess criteria: {success_criteria}\n"
            f"Tools the assistant called, in order: {', '.join(tools_used) or 'none'}\n"
            f"Assistant's reply: {reply}\n"
            "Decide whether the criteria are met, using the tool calls as evidence, and whether the user is needed.")
        print(f"attempt {attempt}: met={verdict.success_criteria_met} "
              f"user_needed={verdict.user_input_needed} tools={tools_used}\n  feedback: {verdict.feedback}")
        if verdict.success_criteria_met or verdict.user_input_needed:
            break
        message = f"Your last response did not meet the success criteria. Feedback: {verdict.feedback}. Keep working."
    return reply

reply = run_turn(
    "Find out who won the Nobel Prize in Physics in 2023 and send me a push notification with a one-line summary.",
    "The reply names the winners, and a push notification was sent.",
)
display(Markdown(reply))

## Pattern recap

| Pattern | Key idea |
|---|---|
| Chat model | `invoke` / `stream`; messages as objects or dicts; any OpenAI-compatible `base_url` |
| `@tool` + `bind_tools` | Schema from signature + docstring; at layer 1 you run the loop |
| `with_structured_output` | Pydantic object back instead of text |
| State + reducer | `TypedDict` flows through nodes; `add_messages` appends instead of replacing |
| 5 steps to a graph | State → builder → nodes → edges → compile |
| `ToolNode` + `tools_condition` | The agent loop as a graph; conditional edges route |
| Checkpointer + `thread_id` | Snapshot per super-step: memory, persistence, inspection, time travel |
| `create_agent` | Layer 3: the tool-loop graph in one line; `response_format`, `checkpointer` |
| Middleware | Your code around model/tool calls: logging, limits, PII, todos, HITL |
| Human-in-the-loop | `interrupt` + `Command(resume=approve/edit/reject)` |
| Deep Agents | Planning + filesystem + sub-agents with clean context |
| Evaluator loop | Worker + structured-output judge with tool calls as evidence, retry with feedback |

Compared to `3_crewai`: LangGraph gives **explicit control flow** (you draw the graph), CrewAI gives
**roles and processes**; `create_agent` sits in between, like the OpenAI Agents SDK in `2_openai`.
For the paid originals, see `1_lab1` → `5_lab5` in this folder.